In [1]:
import numpy as np
import pandas as pd
import ast

In [2]:
original_matrix = pd.read_csv("../result/aou_icd_260217/prs_adjauc_matrix_260217_rootcode_qc.csv")
original_matrix = original_matrix.set_index("trait")

rank_matrix = pd.read_csv("../result/aou_icd_260217/prs_adjauc_matrix_260217_rootcode_rankmatrix.csv")
rank_matrix = rank_matrix.set_index("trait")

target_rank_matrix = pd.read_csv("../result/aou_icd_260217/prs_adjauc_matrix_260217_rootcode_selfrankmatrix.csv")
target_rank_matrix = target_rank_matrix.set_index("trait")

metainfo = pd.concat([
    pd.read_csv("../disease_preprocess/pgs_metadata_260217.csv", encoding='latin-1'),
    pd.read_csv("../measurement_preprocess/pgs_metadata_260225.csv", encoding='latin-1')
], ignore_index=True)
metainfo = metainfo.drop_duplicates(subset="PGS_ID", keep="first").set_index("PGS_ID")

In [3]:
simple_keywords = [
    # GWAS significance selection
    "gwas", "genome-wide significant", "genomewide-significant",
    "significant snp", "significant variant", "significant loci",
    # Clumping & thresholding
    "clump", "threshold", "c+t", "p+t", "pruning", "prunin", "thinning",
    "plink", "prsice", "ricopilli",
    # Curated / literature selection
    "curated", "literature", "known", "previously associated",
    "established", "candidate", "representative", "suggestive",
    "lead snp", "lead variant", "fine-mapping",
    # Simple regression without regularization
    "stepwise", "hard threshold", "hazard model",
    "log-or weighted", "composite likelihood ratio",
    "likelihood ratio",
]

advanced_keywords = [
    # LDPred family
    "ldpred", "ldpred2",
    # PRS-CS family
    "prs-cs", "prscs", "prscsx", "prs-csx",
    # Lassosum family
    "lassosum",
    # SBayes family
    "sbayesr", "sbayesrc", "sbayess",
    # Other Bayesian
    "bayesian", "bayes", "annopred", "polyfun", "dbslmm", "sdpr",
    # Penalized regression
    "lasso", "ridge", "elastic", "bigstatsr", "biglasso", "snpnet",
    "sparsnp", "penalized", "regularized",
    # Ensemble / meta
    "metagrs", "metaprs", "prsmix", "prssum", "prosper", "shprs",
    "stacked", "sct", "ct-sleb", "ctsleb",
    # Other ML
    "geboost", "genoboost", "bolt-lmm", "boltlmm",
    "megaprs", "pt_clump", "weighted_ldpred", "weighted_lassosum",
]

In [55]:
today = pd.Timestamp.today()
rows = []

for trait in target_rank_matrix.index:
    # select trait that have target PGS more than 10 
    target_pgs_series = target_rank_matrix.loc[trait].dropna().tolist()
    performance_series = original_matrix.loc[trait, target_pgs_series].dropna()
    n_target_pgs = len(performance_series)
    if n_target_pgs < 10:
        continue
    ranked = performance_series.rank(ascending=False)

    # construct feature map
    for col, abs_rank in ranked.items():
        pgs_id = col.split("__")[1]
        if pgs_id not in metainfo.index:
            continue
        meta = metainfo.loc[pgs_id]

        # training feature
        num_var   = pd.to_numeric(meta.get("num_variant", np.nan), errors='coerce')
        num_train = pd.to_numeric(meta.get("num_training_sample", np.nan), errors='coerce')
        num_eval  = pd.to_numeric(meta.get("num_eval", np.nan), errors='coerce')
        try:
            days_since = (today - pd.to_datetime(meta["date_release"])).days
        except:
            days_since = np.nan

        # training_ancestry
        ancestry    = str(meta.get("training_ancestry", "")).lower()
        trained_w_european = int("european" in ancestry)

        # training_method
        method = str(meta.get("training_method", "")).lower()
        is_advanced = int(any(k in method for k in advanced_keywords))

        # eval features
        try:
            eval_est  = ast.literal_eval(meta["eval_estimate"])
            eval_anc  = ast.literal_eval(meta["eval_ancestry"])
            eval_met  = ast.literal_eval(meta["eval_metrics"])
            eval_cov  = ast.literal_eval(meta["eval_covariates"])
            eval_coh = ast.literal_eval(meta["eval_cohort"])
            eval_samp = ast.literal_eval(meta["num_eval_sample"])

            cov_filter = (
                lambda c: "age" in str(c).lower()
                and ("sex" in str(c).lower() or "gender" in str(c).lower())
                and ("pc" in str(c).lower() or "principal" in str(c).lower() or "ancestr" in str(c).lower())
            )

            # AUC pairs (preferred)
            auc_pairs = [
                (float(e), str(m_).lower(), c, s, co)
                for e, a, m_, c, s, co in zip(eval_est, eval_anc, eval_met, eval_cov, eval_samp, eval_coh)
                if a == 'European' and e is not None
                and any(k in str(m_).lower() for k in ["auc", "auroc", "area under"])
                and cov_filter(c)
            ]

            # partial R² fallback
            r2_pairs = [
                (float(e), str(m_).lower(), c, s, co)
                for e, a, m_, c, s, co in zip(eval_est, eval_anc, eval_met, eval_cov, eval_samp, eval_coh)
                if a == 'European' and e is not None
                and any(k in str(m_).lower() for k in ["partial"])
                and cov_filter(c)
            ]

            # incremental r2 fallback
            increr2_pairs = [
                (float(e), str(m_).lower(), c, s, co)
                for e, a, m_, c, s, co in zip(eval_est, eval_anc, eval_met, eval_cov, eval_samp, eval_coh)
                if a == 'European' and e is not None
                and any(k in str(m_).lower() for k in ["incremental"])
                and cov_filter(c)
            ]

            # Odds Ratio fallback
            odds_pairs = [
                (float(e), str(m_).lower(), c, s, co)
                for e, a, m_, c, s, co in zip(eval_est, eval_anc, eval_met, eval_cov, eval_samp, eval_coh)
                if a == 'European' and e is not None
                and any(k in str(m_).lower() for k in ["odds"])
                and cov_filter(c)
            ]

            # Hazard Ratio fallback
            hazard_pairs = [
                (float(e), str(m_).lower(), c, s, co)
                for e, a, m_, c, s, co in zip(eval_est, eval_anc, eval_met, eval_cov, eval_samp, eval_coh)
                if a == 'European' and e is not None
                and any(k in str(m_).lower() for k in ["hazard"])
                and cov_filter(c)
            ]

            def best_pair(pairs):
                return max(pairs, key=lambda p: float(p[3]) if p[3] else 0)

            if auc_pairs:
                bp = best_pair(auc_pairs)
                eval_metric_type = 1
            elif r2_pairs:
                bp = best_pair(r2_pairs)
                eval_metric_type = 2
            elif increr2_pairs:
                bp = best_pair(increr2_pairs)
                eval_metric_type = 3
            elif odds_pairs:
                bp = best_pair(odds_pairs)
                eval_metric_type = 4
            elif hazard_pairs:
                bp = best_pair(hazard_pairs)
                eval_metric_type = 5
            else:
                bp = None
                eval_metric_type = None

            if bp:
                reported_eval    = bp[0]
                eval_sample_size = bp[3]   # ← 注意：原代码用的[4]是cohort，应该是[3]
                co = str(bp[4]).lower()
            else:
                reported_eval    = np.nan
                eval_sample_size = np.nan
                co = ""

            if "uk biobank" in co or "ukbb" in co or "ukb" in co:
                eval_cohort = 1
            elif "all of us" in co or "aou" in co:
                eval_cohort = 2
            elif "million veteran" in co or "mvp" in co:
                eval_cohort = 3
            elif "finngen" in co:
                eval_cohort = 4
            else:
                eval_cohort = 0  # other / missing

        except:
            reported_eval = np.nan
            eval_metric_type = None


        pct_rank = 1 - (abs_rank - 1) / (n_target_pgs - 1)
        # top 50% are selected as 1
        is_top_50percent = int(abs_rank <= n_target_pgs / 2)
        # top 10% are selected as 1
        is_top_10percent = int(abs_rank <= max(1, round(n_target_pgs * 0.1)))
        # only the best is selected as 1
        is_best = int(abs_rank == 1)

        rows.append({
            "trait": trait, "pgs_id": pgs_id,
            "num_variant":         num_var,
            "num_training_sample": num_train,
            "num_eval":            num_eval,
            "days_since_release":  days_since,
            "training_w_european": trained_w_european,
            "advanced_method":     is_advanced,
            "eval_metric_type":    eval_metric_type,
            "reported_eval":       reported_eval,
            "eval_sample_size":    eval_sample_size,
            "eval_cohort":         eval_cohort,
            "pct_rank":            pct_rank,
            "is_top_50percent":    is_top_50percent,
            "is_top_10percent":    is_top_10percent,
            "is_best":             is_best
        })

df_glm = pd.DataFrame(rows)

features = [
    "num_variant", "num_training_sample", "num_eval",
    "days_since_release", "training_w_european", "advanced_method",
    "eval_metric_type", "reported_eval", "eval_sample_size", "eval_cohort"
]

print(f"Total samples: {len(df_glm)}")
print(f"\nMissing rate per feature:")
print(df_glm[features].isna().mean().round(3).to_string())

df_glm.to_csv("pgs_features.csv", index=False)
print(f"Saved {len(df_glm)} rows to pgs_features.csv")

Total samples: 2093

Missing rate per feature:
num_variant            0.000
num_training_sample    0.008
num_eval               0.000
days_since_release     0.000
training_w_european    0.000
advanced_method        0.000
eval_metric_type       0.457
reported_eval          0.457
eval_sample_size       0.457
eval_cohort            0.000
Saved 2093 rows to pgs_features.csv


### Binary classification

In [6]:
df_glm = pd.read_csv("pgs_features.csv")
features = [
    "num_variant", "num_training_sample", "num_eval",
    "days_since_release", "training_w_european", "advanced_method",
    "eval_metric_type", "reported_eval", "eval_sample_size", "eval_cohort"
]

In [3]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt

In [4]:
model_dict = {
    "Logistic":          Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=1000))]),
    "Random Forest":     RandomForestClassifier(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, random_state=42),
}
cv = StratifiedKFold(5, shuffle=True, random_state=42)

In [7]:
# log transform 偏态列
log_cols = ["num_variant", "num_training_sample", "eval_sample_size"]
df_glm[log_cols] = np.log1p(df_glm[log_cols].astype(float))

for t in ["is_top_50percent", "is_top_10percent", "is_best"]:
    print(f"----------{t}----------")
    df_clean = df_glm[features + [t]].dropna()
    print(f"Samples: {len(df_clean)}")
    print(f"Dropped: {len(df_glm) - len(df_clean)}")

    X = df_clean[features].values.astype(float)
    y = df_clean[t].values

    results = {}
    for name, model in model_dict.items():
        scores = cross_validate(model, X, y, cv=cv,
                            scoring={'auc': 'roc_auc', 'f1': 'f1'})
        results[name] = scores
        print(f"{name:<20} AUC: {scores['test_auc'].mean():.4f} ± {scores['test_auc'].std():.4f} | "
          f"F1: {scores['test_f1'].mean():.4f} ± {scores['test_f1'].std():.4f}")

----------is_top_50percent----------
Samples: 1136
Dropped: 957
Logistic             AUC: 0.6680 ± 0.0237 | F1: 0.6295 ± 0.0299
Random Forest        AUC: 0.8322 ± 0.0174 | F1: 0.7532 ± 0.0250
Gradient Boosting    AUC: 0.8353 ± 0.0311 | F1: 0.7575 ± 0.0283
----------is_top_10percent----------
Samples: 1136
Dropped: 957
Logistic             AUC: 0.6874 ± 0.0786 | F1: 0.0000 ± 0.0000
Random Forest        AUC: 0.8143 ± 0.0676 | F1: 0.3685 ± 0.0661
Gradient Boosting    AUC: 0.7896 ± 0.0743 | F1: 0.2546 ± 0.0695
----------is_best----------
Samples: 1136
Dropped: 957
Logistic             AUC: 0.6766 ± 0.0761 | F1: 0.0000 ± 0.0000
Random Forest        AUC: 0.7578 ± 0.0996 | F1: 0.1238 ± 0.1524
Gradient Boosting    AUC: 0.6205 ± 0.0590 | F1: 0.2116 ± 0.1835


### Regression

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import cross_val_score, cross_val_predict, KFold
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr
from sklearn.neural_network import MLPRegressor

In [20]:
!pip install xgboost lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 27.7 MB/s eta 0:00:00


In [18]:
model_dict = {
    "Linear (Ridge)":    Pipeline([("scaler", StandardScaler()), ("reg", Ridge())]),
    "Random Forest":     RandomForestRegressor(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, random_state=42),
    "Neural Network":    Pipeline([
        ("scaler", StandardScaler()),
        ("reg", MLPRegressor(
            hidden_layer_sizes=(32, 16),
            activation='relu',
            max_iter=500,
            random_state=42,
            early_stopping=True,
            validation_fraction=0.1,
        ))
    ]),
}
cv = KFold(5, shuffle=True, random_state=42)

In [19]:
# log transform 偏态列
log_cols = ["num_variant", "num_training_sample", "eval_sample_size"]
df_glm[log_cols] = np.log1p(df_glm[log_cols].astype(float))

t = "pct_rank"
print(f"----------{t}----------")
df_clean = df_glm[features + [t]].dropna()
print(f"Samples: {len(df_clean)}")
print(f"Dropped: {len(df_glm) - len(df_clean)}")

X = df_clean[features].values.astype(float)
y = df_clean[t].values

results = {}
for name, model in model_dict.items():
    r2  = cross_val_score(model, X, y, cv=cv, scoring='r2').mean()
    mae = -cross_val_score(model, X, y, cv=cv, scoring='neg_mean_absolute_error').mean()

    y_pred = cross_val_predict(model, X, y, cv=cv)
    rho = spearmanr(y, y_pred).statistic
    
    results[name] = {'r2': r2, 'mae': mae, 'rho': rho}
    print(f"{name:<20} R2: {r2:>8.4f} | MAE: {mae:>8.4f} | Rho: {rho:>8.4f}")

----------pct_rank----------
Samples: 1136
Dropped: 957
Linear (Ridge)       R2:   0.0864 | MAE:   0.2414 | Rho:   0.3088
Random Forest        R2:   0.4003 | MAE:   0.1771 | Rho:   0.6329
Gradient Boosting    R2:   0.4199 | MAE:   0.1770 | Rho:   0.6452
Neural Network       R2:   0.1190 | MAE:   0.2302 | Rho:   0.3872


In [ ]:
# 取 spearman 最高的 top 6 traits
top_traits = per_trait_sorted.tail(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

for i, trait in enumerate(top_traits):
    g = df_clean[df_clean["trait"] == trait]
    rho = per_trait_sorted.loc[trait, "spearman"]

    axes[i].scatter(g["y_cont"], g["y_pred"], alpha=0.7, edgecolors='white', linewidth=0.3)

    # 趋势线
    z = np.polyfit(g["y_cont"], g["y_pred"], 1)
    x_line = np.linspace(g["y_cont"].min(), g["y_cont"].max(), 100)
    axes[i].plot(x_line, np.poly1d(z)(x_line), color='red', linewidth=1.2, linestyle='--')

    axes[i].set_xlabel("True pct_rank", fontsize=10)
    axes[i].set_ylabel("Predicted pct_rank", fontsize=10)
    axes[i].set_title(f"{trait}  (ρ={rho:.3f})", fontsize=11)

plt.suptitle("Top 6 Traits by Spearman ρ (GradientBoosting, CV)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()